In [2]:
# build_transformative_books_I_II.py
# Generates: Transformative_Elements_Books_I_II_Compact.pdf
# Requirements: pip install reportlab matplotlib numpy
pip install reportlab matplotlib numpy
python build_transformative_books_I_II.py
# => outputs Transformative_Elements_Books_I_II_Compact.pdf
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for headless render
import matplotlib.pyplot as plt
from matplotlib.patches import Arc, Rectangle

from reportlab.lib.pagesizes import LETTER
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak, Image, Preformatted
)
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY
from reportlab.lib.units import inch


# ======================
# Figure helpers
# ======================

def _savefig_medium(path, dpi=140):
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()


def fig_equilateral(path):
    A = np.array([0.0, 0.0])
    B = np.array([1.0, 0.0])
    r = np.linalg.norm(B - A)
    C = np.array([0.5, np.sqrt(max(r**2 - 0.5**2, 0.0))])  # safe sqrt
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.add_patch(plt.Circle(A, r, fill=False, linestyle="--"))
    ax.add_patch(plt.Circle(B, r, fill=False, linestyle="--"))
    ax.plot([A[0], B[0], C[0], A[0]], [A[1], B[1], C[1], A[1]], lw=2)
    ax.scatter([A[0], B[0], C[0]], [A[1], B[1], C[1]])
    ax.text(A[0]-0.05, A[1]-0.05, "A")
    ax.text(B[0]+0.02, B[1]-0.05, "B")
    ax.text(C[0], C[1]+0.02, "C")
    ax.set_aspect("equal")
    ax.set_xlim(-0.2, 1.2)
    ax.set_ylim(-0.2, 1.0)
    ax.axis("off")
    _savefig_medium(path)


def fig_isosceles(path):
    A = np.array([0.0, 0.0])
    B = np.array([1.0, 0.0])
    C = np.array([0.5, 0.9])
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.plot([A[0], B[0], C[0], A[0]], [A[1], B[1], C[1], A[1]], lw=2)
    ax.scatter([A[0], B[0], C[0]], [A[1], B[1], C[1]])
    ax.text(A[0]-0.05, A[1]-0.05, "A")
    ax.text(B[0]+0.02, B[1]-0.05, "B")
    ax.text(C[0]-0.02, C[1]+0.02, "C")
    arcA = Arc((A[0], A[1]), 0.35, 0.35, angle=0, theta1=0, theta2=60)
    arcB = Arc((B[0], B[1]), 0.35, 0.35, angle=0, theta1=120, theta2=180)
    ax.add_patch(arcA)
    ax.add_patch(arcB)
    ax.text(A[0]+0.2, A[1]+0.08, "angle A")
    ax.text(B[0]-0.3, B[1]+0.08, "angle B")
    ax.set_aspect("equal")
    ax.set_xlim(-0.2, 1.2)
    ax.set_ylim(-0.2, 1.0)
    ax.axis("off")
    _savefig_medium(path)


def fig_pythagoras(path):
    A = np.array([0.0, 0.0])
    B = np.array([1.0, 0.0])     # base = 1
    C = np.array([0.0, 0.6])     # height = 0.6, hypo ~ 1.166
    fig, ax = plt.subplots(figsize=(4, 3))
    # triangle
    ax.plot([A[0], B[0], C[0], A[0]], [A[1], B[1], C[1], A[1]], lw=2)
    ax.scatter([A[0], B[0], C[0]], [A[1], B[1], C[1]])
    ax.text(A[0]-0.05, A[1]-0.05, "A")
    ax.text(B[0]+0.02, B[1]-0.05, "B")
    ax.text(C[0]-0.05, C[1]+0.02, "C")
    # squares on sides
    ax.add_patch(Rectangle((0, 0), 1.0, 1.0, fill=False, linestyle="--"))      # on AB
    ax.add_patch(Rectangle((-0.6, 0), 0.6, 0.6, fill=False, linestyle="--"))   # on AC
    # rotated square on BC
    v = C - B
    L = np.linalg.norm(v)
    if L > 1e-9:
        e = v / L
        n = np.array([-e[1], e[0]])
        P0 = B
        P1 = B + v
        P2 = P1 + n * L
        P3 = B + n * L
        ax.plot([P0[0], P1[0], P2[0], P3[0], P0[0]], [P0[1], P1[1], P2[1], P3[1], P0[1]], linestyle="--")
    ax.set_aspect("equal")
    ax.set_xlim(-0.8, 1.4)
    ax.set_ylim(-0.3, 1.4)
    ax.axis("off")
    _savefig_medium(path)


def fig_square_sum(path, a=0.6, b=0.35):
    fig, ax = plt.subplots(figsize=(4, 3))
    s = a + b
    ax.add_patch(Rectangle((0, 0), s, s, fill=False, lw=2))
    ax.add_patch(Rectangle((0, 0), a, a, fill=False, linestyle="--"))
    ax.add_patch(Rectangle((a, 0), b, a, fill=False, linestyle="--"))
    ax.add_patch(Rectangle((0, a), a, b, fill=False, linestyle="--"))
    ax.add_patch(Rectangle((a, a), b, b, fill=False, linestyle="--"))
    ax.text(a/2, -0.05, "a", ha="center", va="top")
    ax.text(a + b/2, -0.05, "b", ha="center", va="top")
    ax.text(-0.05, a/2, "a", ha="right", va="center", rotation=90)
    ax.text(-0.05, a + b/2, "b", ha="right", va="center", rotation=90)
    ax.text(a/2, a/2, "a^2", ha="center", va="center")
    ax.text(a + b/2, a/2, "ab", ha="center", va="center")
    ax.text(a/2, a + b/2, "ab", ha="center", va="center")
    ax.text(a + b/2, a + b/2, "b^2", ha="center", va="center")
    ax.set_aspect("equal")
    ax.set_xlim(-0.2, s + 0.2)
    ax.set_ylim(-0.2, s + 0.2)
    ax.axis("off")
    _savefig_medium(path)


def fig_diff_squares(path, a=0.8, b=0.45):
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.add_patch(Rectangle((0, 0), a, a, fill=False, lw=2))
    ax.add_patch(Rectangle((0, 0), b, b, fill=False, lw=1, linestyle="--"))
    ax.plot([b, a, a, b, b], [0, 0, a-b, a-b, 0], linestyle="-.")
    ax.text((a+b)/2, (a-b)/2, "(a-b)*(a+b)", ha="center", va="center", fontsize=9)
    ax.text(a/2, a/2, "a^2", ha="center", va="center")
    ax.text(b/2, b/2, "b^2", ha="center", va="center")
    ax.set_aspect("equal")
    ax.set_xlim(-0.2, a + 0.2)
    ax.set_ylim(-0.2, a + 0.2)
    ax.axis("off")
    _savefig_medium(path)


# ======================
# PDF builder
# ======================

def build_pdf(out_path):
    # Prepare figure assets
    asset_dir = os.path.dirname(out_path) or "."
    figs = {
        "equilateral": os.path.join(asset_dir, "fig_equilateral.png"),
        "isosceles": os.path.join(asset_dir, "fig_isosceles.png"),
        "pythagoras": os.path.join(asset_dir, "fig_pythagoras.png"),
        "square_sum": os.path.join(asset_dir, "fig_square_sum.png"),
        "diff_squares": os.path.join(asset_dir, "fig_diff_squares.png"),
    }
    fig_equilateral(figs["equilateral"])
    fig_isosceles(figs["isosceles"])
    fig_pythagoras(figs["pythagoras"])
    fig_square_sum(figs["square_sum"])
    fig_diff_squares(figs["diff_squares"])

    # Styles
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="TitleX", alignment=TA_CENTER, fontSize=16, leading=20, spaceAfter=8))
    styles.add(ParagraphStyle(name="SubX", alignment=TA_CENTER, fontSize=11, leading=14, textColor=colors.grey))
    styles.add(ParagraphStyle(name="H1X", fontSize=13, leading=17, spaceBefore=10, spaceAfter=4, textColor=colors.darkblue))
    styles.add(ParagraphStyle(name="H2X", fontSize=11.5, leading=15, spaceBefore=6, spaceAfter=2, textColor=colors.darkgreen))
    styles.add(ParagraphStyle(name="BodyX", fontSize=9.8, leading=13, alignment=TA_JUSTIFY))
    styles.add(ParagraphStyle(name="CodeMonoX", fontName="Courier", fontSize=8.6, leading=11))
    styles.add(ParagraphStyle(name="EqX", fontSize=10.0, leading=13, leftIndent=12))

    # Doc
    doc = SimpleDocTemplate(
        out_path, pagesize=LETTER,
        leftMargin=0.6 * inch, rightMargin=0.6 * inch, topMargin=0.6 * inch, bottomMargin=0.6 * inch
    )
    story = []

    # Title
    story.append(Paragraph("Transformative Elements — Books I & II (Educational Compact Edition)", styles["TitleX"]))
    story.append(Paragraph("Dynamic Geometry and Algebra with pi(), phi(), C(), I()", styles["SubX"]))
    story.append(Spacer(1, 0.1 * inch))

    # BOOK I
    story.append(Paragraph("BOOK I — Transformative Geometry (Seeds of Awareness)", styles["H1X"]))
    story.append(Paragraph(
        "Point D0 as seed; Line D1 as influence I(); Plane D2 where curvature pi() matters; Form D3 stabilized by awareness C(). "
        "We retain Euclid's rigor and add living operators: pi() for curvature, phi() for harmonic proportion, I() for influence, "
        "C() for awareness/constraints.", styles["BodyX"]))

    # Operators (compact)
    story.append(Paragraph("Operators (compact definitions for Book I–II):", styles["H2X"]))

    for o in [
        "pi(figure): local curvature measure; in fields, pi(u)=Δu; in pure geometry, pi=0 along straight relations.",
        "phi(A,B,...): proportion operator; at equilibrium phi=1 (balanced influence, equal ratios).",
        "I(figure or field): influence (energy of relation); for D2 field u, I=∑(ux^2+uy^2). For segments, I is length; for areas, I parallels area.",
        "C(context): awareness/constraint that selects a target (e.g., choose phi*=1) and enforces boundary or symmetry during construction."
    ]:
        story.append(Paragraph("• " + o, styles["BodyX"]))

    # I.1
    story.append(Paragraph("I.1 — Equilateral Triangle on AB", styles["H1X"]))
    story.append(Paragraph(
        "Construction: circles of radius |AB| centered at A and B intersect at C. Then AC=BC=AB. "
        "Transformative: equalize influence from two centers so phi across edges = 1.", styles["BodyX"]))
    story.append(Image(figs["equilateral"], width=3.6*inch, height=2.7*inch))
    story.append(Paragraph("ASCII fallback:", styles["H2X"]))
    story.append(Preformatted(r"""
A o----r----o B
 \          /
  \        /
   \  C   /
    \    /
     \  /
      \/
""", styles["CodeMonoX"]))

    # I.5
    story.append(Paragraph("I.5 — Base Angles in an Isosceles Triangle are Equal", styles["H1X"]))
    story.append(Paragraph(
        "Given AB=AC, mirror-awareness C() preserves curvature exposure at the base, so ∠A = ∠B. "
        "Proof: reflect across altitude; base endpoints swap and arcs remain equal.", styles["BodyX"]))
    story.append(Image(figs["isosceles"], width=3.6*inch, height=2.7*inch))

    # I.47
    story.append(Paragraph("I.47 — Pythagorean Theorem (Influence Conservation)", styles["H1X"]))
    story.append(Paragraph(
        "In right triangle ABC, areas on legs sum to area on hypotenuse. Transformative: orthogonal influences add: I_x + I_y = I_h. "
        "Euclid's squares and the influence view agree.", styles["BodyX"]))
    story.append(Image(figs["pythagoras"], width=3.6*inch, height=2.7*inch))

    story.append(PageBreak())

    # BOOK II
    story.append(Paragraph("BOOK II — Transformative Algebra of Space", styles["H1X"]))
    story.append(Paragraph(
        "Algebraic identities are geometric energy laws: area as stored influence; rearrangements conserve totals. "
        "Phi() regulates proportion; C() selects targets.", styles["BodyX"]))

    # (a+b)^2
    story.append(Paragraph("(a + b)^2 = a^2 + 2ab + b^2 — Influence Overlap", styles["H2X"]))
    story.append(Image(figs["square_sum"], width=3.6*inch, height=2.7*inch))
    story.append(Preformatted(r"""
+---------+
| a^2 |ab |
|-----+---|  (a+b)^2 partition
| ab  |b^2|
+---------+
""", styles["CodeMonoX"]))
    story.append(Paragraph("Interpretation: I_total = I_a + I_b + 2*I_cross. The cross-terms represent shared influence between subfields.",
                          styles["BodyX"]))

    # a^2 - b^2
    story.append(Paragraph("a^2 - b^2 = (a - b)(a + b) — Compression/Expansion", styles["H2X"]))
    story.append(Image(figs["diff_squares"], width=3.6*inch, height=2.7*inch))
    story.append(Preformatted(r"""
Outer: a^2, Inner removed: b^2
Remaining area forms rectangle (a-b) by (a+b).
""", styles["CodeMonoX"]))

    # Closing
    story.append(Paragraph("C(), phi(), and Applications", styles["H2X"]))
    story.append(Paragraph(
        "C() acts as awareness/constraint (e.g., choose phi*=1 to enforce balanced proportion). "
        "Phi() regulates proportional structure; pi() governs curvature; I() measures relation cost. "
        "Together they produce a living geometry: teachable by constructions, provable by invariants, "
        "realizable in code across physics, AI, and biology.", styles["BodyX"]))

    doc.build(story)


def main():
    out = "Transformative_Elements_Books_I_II_Compact.pdf"
    build_pdf(out)
    print(f"Built: {out}")


if __name__ == "__main__":
    main()

SyntaxError: invalid syntax (ipython-input-1284468003.py, line 4)